<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OHDSI FHIR-to-OMOP Reviewer-Driven Hardening — Google Colab

**Project:** *A Lightweight FHIR-to-OMOP Software Demonstration Using pyOMOP*

This notebook executes the reviewer-driven hardening plan while preserving the accepted lightweight educational/software-demo scope.

### Reviewer-critical questions
1. Why did **25 synthetic FHIR Patient resources** correspond to **27 OMOP `person` rows**?
2. Why was `drug_exposure.drug_concept_id` mapping **78.12%**?
3. Why were **1,386 `visit_occurrence` rows** created while non-zero `visit_concept_id` mapping was **0.00%**?
4. What is the practical execution time on the recorded environment?
5. Can the workflow produce reproducible automated checks?

> **Important:** 0.00% visit concept mapping does **not** mean zero visit rows. Row creation and terminology mapping are separate outcomes.

### Accepted-submission reference values
These are reference values only. The notebook recomputes results from the actual files/database.

| Metric | Reference |
|---|---:|
| FHIR Patient resources | 25 |
| OMOP `person` | 27 |
| `visit_occurrence` | 1,386 |
| `condition_occurrence` | 983 |
| `drug_exposure` | 1,275 |
| `observation` | 14,168 |
| `measurement` | 14,150 |
| Condition mapping | 100.00% |
| Drug mapping | 78.12% |
| Measurement mapping | 99.99% |
| Observation mapping | 99.99% |
| Visit concept mapping | 0.00% |

### Output policy
- Aggregate, GitHub-safe results → `results/reviewer_hardening/`
- Row-level IDs/source values/logs → `/content/OHDSI_PRIVATE_REVIEWER_DIAGNOSTICS/`
- Athena vocabulary files are never copied into the repository.

## 0. Install lightweight dependencies and clone/update the repository

The project currently declares `pyomop==6.4.0`; this notebook uses the same version for code-path inspection.

In [1]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyomop==6.4.0", "psutil"],
    check=True,
)

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print("Repository exists; pulling latest main branch...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

print("Repository:", REPO_DIR)

Repository: /content/ohdsi-fhir-omop-showcase-demo


In [2]:
import os, json, time, sqlite3, platform, shutil, re, inspect
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
import psutil
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped outside Colab.")

DATA_ROOT = Path("/content/drive/MyDrive/OHDSI_FHIR_OMOP")
OMOP_DB_PATH = None   # Example: DATA_ROOT / "ohdsi_demo.sqlite"
FHIR_ROOT = None      # Example: DATA_ROOT / "fhir_bulk"

PUBLIC_RESULTS_DIR = REPO_DIR / "results" / "reviewer_hardening"
PRIVATE_RESULTS_DIR = Path("/content/OHDSI_PRIVATE_REVIEWER_DIAGNOSTICS")
PUBLIC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRIVATE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Leave blank until the exact full FHIR->OMOP transformation command is verified.
TRANSFORM_COMMAND = ""

REFERENCE = {
    "fhir_patient_resources": 25,
    "person": 27,
    "visit_occurrence": 1386,
    "condition_occurrence": 983,
    "drug_exposure": 1275,
    "observation": 14168,
    "measurement": 14150,
}
REFERENCE_MAPPING = {
    "condition_occurrence": 100.00,
    "drug_exposure": 78.12,
    "measurement": 99.99,
    "observation": 99.99,
    "visit_occurrence": 0.00,
}

print("DATA_ROOT:", DATA_ROOT)
print("PUBLIC_RESULTS_DIR:", PUBLIC_RESULTS_DIR)
print("PRIVATE_RESULTS_DIR:", PRIVATE_RESULTS_DIR)

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/OHDSI_FHIR_OMOP
PUBLIC_RESULTS_DIR: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening
PRIVATE_RESULTS_DIR: /content/OHDSI_PRIVATE_REVIEWER_DIAGNOSTICS


## 2. Helper functions and auto-discovery

In [3]:
timings = {}
validation_rows = []

class Timer:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc, tb):
        elapsed = time.perf_counter() - self.start
        timings[self.label] = elapsed
        print(f"[timing] {self.label}: {elapsed:.3f} sec")

def table_exists(conn, table):
    return conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name=? LIMIT 1",
        (table,)
    ).fetchone() is not None

def columns_for(conn, table):
    if not table_exists(conn, table):
        return []
    return [r[1] for r in conn.execute(f'PRAGMA table_info("{table}")').fetchall()]

def safe_count(conn, table):
    if not table_exists(conn, table):
        return None
    return int(conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0])

def normalize_identifier(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value).strip()
    if not s:
        return None
    if s.startswith("urn:uuid:"):
        s = s[len("urn:uuid:"):]
    if "/" in s:
        parts = [p for p in s.split("/") if p]
        if len(parts) >= 2 and parts[-2].lower() == "patient":
            s = parts[-1]
    return s.strip() or None

def pct(mapped, total):
    return 100.0 * mapped / total if total else np.nan

def add_validation(test_id, description, observed, expected=None, status=None, note=""):
    if status is None:
        status = "PASS" if expected is None or observed == expected else "REVIEW"
    validation_rows.append({
        "test_id": test_id,
        "description": description,
        "observed": observed,
        "expected_or_reference": expected,
        "status": status,
        "note": note,
    })

def save_public(df, filename):
    path = PUBLIC_RESULTS_DIR / filename
    df.to_csv(path, index=False)
    print("Saved public:", path)
    return path

def save_private(df, filename):
    path = PRIVATE_RESULTS_DIR / filename
    df.to_csv(path, index=False)
    print("Saved PRIVATE:", path)
    return path

def iter_ndjson_files(root):
    if root is None:
        return []
    root = Path(root)
    if root.is_file():
        return [root]
    if not root.exists():
        return []
    files = []
    for pattern in ("*.ndjson", "*.jsonl", "*.ndjson.gz", "*.jsonl.gz"):
        files.extend(root.rglob(pattern))
    return sorted(set(files))

def iter_resources_from_file(path):
    import gzip
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict) and obj.get("resourceType") == "Bundle":
                for entry in obj.get("entry", []) or []:
                    res = entry.get("resource")
                    if isinstance(res, dict):
                        yield res
            elif isinstance(obj, dict) and obj.get("resourceType"):
                yield obj

def flatten_codings(codeable):
    if isinstance(codeable, dict):
        for coding in codeable.get("coding", []) or []:
            if isinstance(coding, dict):
                yield coding

# --- Database auto-discovery ---
db_candidates = []
if OMOP_DB_PATH is not None:
    db_candidates = [Path(OMOP_DB_PATH)]
else:
    for root in [DATA_ROOT, REPO_DIR]:
        if Path(root).exists():
            for pattern in ("*.sqlite", "*.sqlite3", "*.db"):
                db_candidates.extend(Path(root).rglob(pattern))

def db_rank(p):
    name = p.name.lower()
    score = 0
    if "ohdsi_demo" in name: score -= 10
    if "omop" in name: score -= 5
    if "demo" in name: score -= 2
    return (score, len(str(p)))

db_candidates = sorted(set(db_candidates), key=db_rank)
resolved_db = db_candidates[0] if db_candidates else None

# --- FHIR auto-discovery ---
if FHIR_ROOT is not None:
    resolved_fhir_root = Path(FHIR_ROOT)
else:
    resolved_fhir_root = None
    for d in [
        DATA_ROOT / "fhir_bulk",
        DATA_ROOT / "fhir",
        DATA_ROOT,
        REPO_DIR / "data" / "fhir_bulk",
        REPO_DIR / "data",
    ]:
        if iter_ndjson_files(d):
            resolved_fhir_root = d
            break

fhir_files = iter_ndjson_files(resolved_fhir_root)

print("Resolved OMOP DB:", resolved_db)
print("Resolved FHIR root:", resolved_fhir_root)
print("FHIR files found:", len(fhir_files))
if db_candidates:
    print("\nTop DB candidates:")
    for p in db_candidates[:10]:
        print(" -", p)

Resolved OMOP DB: None
Resolved FHIR root: None
FHIR files found: 0


## 3. Capture machine/software environment

This gives context for runtime reporting. It is **not** a production scalability benchmark.

In [4]:
import importlib.metadata as mdpkg

env = {
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "python_version": sys.version.replace("\n", " "),
    "platform": platform.platform(),
    "processor": platform.processor(),
    "logical_cpu_count": psutil.cpu_count(logical=True),
    "physical_cpu_count": psutil.cpu_count(logical=False),
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    "colab": IN_COLAB,
}
for package in ["pyomop", "pandas", "numpy", "psutil"]:
    try:
        env[f"{package}_version"] = mdpkg.version(package)
    except Exception:
        env[f"{package}_version"] = "not_available"

try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    env["gpu"] = gpu.stdout.strip() if gpu.returncode == 0 and gpu.stdout.strip() else "none"
except Exception:
    env["gpu"] = "none"

env_df = pd.DataFrame([env])
display(env_df.T)
save_public(env_df, "environment_summary.csv")

,0
timestamp_utc,2026-08-16T03:11:02.882329+00:00
python_version,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11...."
platform,Linux-6.6.122+-x86_64-with-glibc2.35
processor,x86_64
logical_cpu_count,8
physical_cpu_count,4
ram_gb,50.99
colab,True
pyomop_version,6.4.0
pandas_version,2.2.2


Saved public: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/environment_summary.csv


PosixPath('/content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/environment_summary.csv')

# Part A — PERSON reconciliation

## 4. Inventory FHIR resources and Patient identifiers

The notebook counts Patient resources, unique IDs, and duplicate IDs. It does not assume the cause of the 25→27 discrepancy.

In [5]:
fhir_resource_counts = Counter()
fhir_patient_ids = []

with Timer("fhir_inventory"):
    if not fhir_files:
        print("Skipped: no FHIR files found.")
    else:
        for path in fhir_files:
            for res in iter_resources_from_file(path):
                rt = res.get("resourceType", "UNKNOWN")
                fhir_resource_counts[rt] += 1
                if rt == "Patient":
                    fhir_patient_ids.append(normalize_identifier(res.get("id")))

fhir_patient_ids_clean = [x for x in fhir_patient_ids if x is not None]
fhir_patient_unique = sorted(set(fhir_patient_ids_clean))

resource_counts_df = pd.DataFrame(
    sorted(fhir_resource_counts.items()),
    columns=["resource_type", "count"]
)
display(resource_counts_df)

print("FHIR Patient resources:", len(fhir_patient_ids))
print("Unique non-null Patient IDs:", len(fhir_patient_unique))

save_public(resource_counts_df, "fhir_resource_counts.csv")

if fhir_patient_ids_clean:
    vc = pd.Series(fhir_patient_ids_clean).value_counts()
    dup_df = vc[vc > 1].rename_axis("normalized_patient_id").reset_index(name="occurrences")
    if not dup_df.empty:
        save_private(dup_df, "fhir_duplicate_patient_ids_PRIVATE.csv")

    add_validation(
        "FHIR-001",
        "FHIR Patient resource count vs accepted reference",
        len(fhir_patient_ids),
        REFERENCE["fhir_patient_resources"],
        "PASS" if len(fhir_patient_ids) == REFERENCE["fhir_patient_resources"] else "REVIEW",
        "Reference comparison only."
    )

Skipped: no FHIR files found.
[timing] fhir_inventory: 0.000 sec


,resource_type,count


FHIR Patient resources: 0
Unique non-null Patient IDs: 0
Saved public: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/fhir_resource_counts.csv


## 5. Inspect OMOP PERSON and reconcile `person_source_value`

A final explanation is produced only if source identifiers make the lineage verifiable. If provenance is not present, the notebook labels that as a traceability limitation rather than inventing a cause.

In [6]:
person_df = pd.DataFrame()
person_reconciliation_summary = pd.DataFrame()

if resolved_db and Path(resolved_db).exists():
    with Timer("person_reconciliation"):
        conn = sqlite3.connect(str(resolved_db))
        if not table_exists(conn, "person"):
            print("Table 'person' not found.")
        else:
            person_cols = columns_for(conn, "person")
            select_cols = [c for c in [
                "person_id", "person_source_value", "gender_concept_id",
                "year_of_birth", "month_of_birth", "day_of_birth",
                "gender_source_value", "race_source_value", "ethnicity_source_value"
            ] if c in person_cols]

            person_df = pd.read_sql_query(
                "SELECT " + ", ".join([f'"{c}"' for c in select_cols]) + " FROM person",
                conn
            )
            display(person_df.head())

            person_count = len(person_df)
            add_validation(
                "OMOP-001",
                "OMOP PERSON row count vs accepted reference",
                person_count,
                REFERENCE["person"],
                "PASS" if person_count == REFERENCE["person"] else "REVIEW",
                "Reference comparison only."
            )

            if "person_source_value" in person_df.columns:
                person_df["normalized_person_source_value"] = person_df["person_source_value"].map(normalize_identifier)
                omop_set = set(person_df["normalized_person_source_value"].dropna().astype(str))
                fhir_set = set(fhir_patient_unique)

                matched = sorted(fhir_set & omop_set)
                omop_only = sorted(omop_set - fhir_set)
                fhir_only = sorted(fhir_set - omop_set)

                dup_omop = (
                    person_df.dropna(subset=["normalized_person_source_value"])
                    .groupby("normalized_person_source_value")
                    .size()
                    .reset_index(name="omop_rows")
                    .query("omop_rows > 1")
                )

                person_reconciliation_summary = pd.DataFrame([{
                    "fhir_patient_resources": len(fhir_patient_ids),
                    "fhir_unique_patient_ids": len(fhir_set),
                    "omop_person_rows": person_count,
                    "omop_unique_nonnull_person_source_values": len(omop_set),
                    "matched_unique_ids": len(matched),
                    "omop_only_unique_ids": len(omop_only),
                    "fhir_only_unique_ids": len(fhir_only),
                    "omop_null_person_source_value_rows": int(person_df["normalized_person_source_value"].isna().sum()),
                    "omop_duplicate_person_source_value_groups": len(dup_omop),
                }])
                display(person_reconciliation_summary)
                save_public(person_reconciliation_summary, "person_reconciliation_summary.csv")

                if omop_only:
                    save_private(pd.DataFrame({"omop_only_id": omop_only}), "person_omop_only_ids_PRIVATE.csv")
                if fhir_only:
                    save_private(pd.DataFrame({"fhir_only_id": fhir_only}), "person_fhir_only_ids_PRIVATE.csv")
                if not dup_omop.empty:
                    save_private(dup_omop, "omop_duplicate_person_source_values_PRIVATE.csv")

                status = "PASS" if (fhir_set and fhir_set == omop_set and person_count == len(fhir_set)) else "REVIEW"
                add_validation(
                    "REC-001",
                    "FHIR Patient IDs reconcile to OMOP person_source_value",
                    f"matched={len(matched)}, omop_only={len(omop_only)}, fhir_only={len(fhir_only)}, person_rows={person_count}",
                    "1:1 identifier and row reconciliation",
                    status,
                    "REVIEW requires diagnosis; it does not automatically mean a pyOMOP bug."
                )
            else:
                person_reconciliation_summary = pd.DataFrame([{
                    "fhir_patient_resources": len(fhir_patient_ids),
                    "fhir_unique_patient_ids": len(fhir_patient_unique),
                    "omop_person_rows": person_count,
                    "traceability_status": "person_source_value absent; direct source-ID reconciliation unavailable"
                }])
                display(person_reconciliation_summary)
                save_public(person_reconciliation_summary, "person_reconciliation_summary.csv")
                add_validation(
                    "REC-001",
                    "Direct source-ID reconciliation",
                    "not_possible",
                    "usable person source provenance",
                    "REVIEW",
                    "Do not assign a root cause until transformation provenance is verified."
                )
        conn.close()
else:
    print("Skipped: OMOP database unavailable.")

Skipped: OMOP database unavailable.


## 6. Additional PERSON discrepancy signals

These checks identify signals such as duplicated/blank source identifiers and repeated demographic signatures. They are **not treated as root causes**.

In [7]:
if not person_df.empty:
    diag_rows = [{"diagnostic": "person_rows", "value": len(person_df), "interpretation": "Total OMOP PERSON rows"}]

    if "person_source_value" in person_df.columns:
        norm = person_df["person_source_value"].map(normalize_identifier)
        diag_rows += [
            {
                "diagnostic": "nonnull_unique_person_source_values",
                "value": int(norm.dropna().nunique()),
                "interpretation": "Unique usable source identifiers"
            },
            {
                "diagnostic": "null_or_blank_person_source_values",
                "value": int(norm.isna().sum()),
                "interpretation": "Rows lacking usable source provenance"
            },
        ]

    sig_cols = [c for c in ["gender_concept_id", "year_of_birth", "month_of_birth", "day_of_birth"] if c in person_df.columns]
    if sig_cols:
        sig_counts = (
            person_df.groupby(sig_cols, dropna=False)
            .size().reset_index(name="rows")
            .sort_values("rows", ascending=False)
        )
        repeated = sig_counts[sig_counts["rows"] > 1]
        diag_rows.append({
            "diagnostic": "repeated_demographic_signature_groups",
            "value": len(repeated),
            "interpretation": "Signal only; matching demographics do not prove duplicate patients"
        })
        if not repeated.empty:
            save_private(repeated, "person_repeated_demographic_signatures_PRIVATE.csv")

    person_diag_df = pd.DataFrame(diag_rows)
    display(person_diag_df)
    save_public(person_diag_df, "person_discrepancy_diagnostic_signals.csv")
else:
    print("Skipped: PERSON data unavailable.")

Skipped: PERSON data unavailable.


# Part B — Mapping completeness and root-cause evidence

## 7. Recompute mapping completeness directly from OMOP

In [8]:
MAPPING_CONFIG = {
    "condition_occurrence": ("condition_concept_id", "condition_source_value", "condition_source_concept_id"),
    "drug_exposure": ("drug_concept_id", "drug_source_value", "drug_source_concept_id"),
    "measurement": ("measurement_concept_id", "measurement_source_value", "measurement_source_concept_id"),
    "observation": ("observation_concept_id", "observation_source_value", "observation_source_concept_id"),
    "visit_occurrence": ("visit_concept_id", "visit_source_value", "visit_source_concept_id"),
}

mapping_rows = []
mapping_df = pd.DataFrame()

if resolved_db and Path(resolved_db).exists():
    with Timer("mapping_completeness"):
        conn = sqlite3.connect(str(resolved_db))

        for table, (concept_col, source_value_col, source_concept_col) in MAPPING_CONFIG.items():
            if not table_exists(conn, table):
                mapping_rows.append({
                    "omop_table": table, "concept_column": concept_col,
                    "status": "table_missing"
                })
                continue

            cols = columns_for(conn, table)
            if concept_col not in cols:
                mapping_rows.append({
                    "omop_table": table, "concept_column": concept_col,
                    "total_records": safe_count(conn, table),
                    "status": "concept_column_missing"
                })
                continue

            total, mapped = conn.execute(
                f'''SELECT COUNT(*),
                           SUM(CASE WHEN "{concept_col}" IS NOT NULL AND "{concept_col}" <> 0 THEN 1 ELSE 0 END)
                    FROM "{table}"'''
            ).fetchone()

            total = int(total or 0)
            mapped = int(mapped or 0)
            unmapped = total - mapped
            mp = pct(mapped, total)
            ref = REFERENCE_MAPPING.get(table)

            mapping_rows.append({
                "omop_table": table,
                "concept_column": concept_col,
                "total_records": total,
                "mapped_records": mapped,
                "unmapped_records": unmapped,
                "mapped_percent": round(mp, 4),
                "accepted_reference_percent": ref,
                "difference_from_reference_pp": round(mp - ref, 4) if ref is not None else np.nan,
                "status": "ok",
            })

            add_validation(
                f"MAP-{table}",
                f"{table}.{concept_col} mapped percent vs accepted reference",
                round(mp, 2),
                ref,
                "PASS" if round(mp, 2) == round(ref, 2) else "REVIEW",
                "Reference comparison only; justified corrections may change the result."
            )

        conn.close()

mapping_df = pd.DataFrame(mapping_rows)
display(mapping_df)
if not mapping_df.empty:
    save_public(mapping_df, "mapping_completeness_recomputed.csv")

""


## 8. Generate the precise visit statement and investigate drug/visit gap signals

The public output reports aggregate diagnostics. Source values and concept-level details are kept private.

In [9]:
visit_statement_df = pd.DataFrame()
gap_public_rows = []

if not mapping_df.empty:
    vr = mapping_df[mapping_df["omop_table"] == "visit_occurrence"]
    if not vr.empty and pd.notna(vr.iloc[0].get("total_records")):
        r = vr.iloc[0]
        visit_statement_df = pd.DataFrame([{
            "visit_occurrence_rows": int(r["total_records"]),
            "rows_with_nonzero_visit_concept_id": int(r["mapped_records"]),
            "rows_without_nonzero_visit_concept_id": int(r["unmapped_records"]),
            "visit_concept_mapping_percent": float(r["mapped_percent"]),
            "recommended_interpretation":
                f"The pipeline generated {int(r['total_records']):,} visit_occurrence rows; "
                f"{float(r['mapped_percent']):.2f}% contained a non-zero visit_concept_id."
        }])
        display(visit_statement_df)
        save_public(visit_statement_df, "visit_mapping_statement_ready_summary.csv")

if resolved_db and Path(resolved_db).exists():
    with Timer("mapping_gap_diagnostics"):
        conn = sqlite3.connect(str(resolved_db))
        has_concept = table_exists(conn, "concept")

        for table in ["drug_exposure", "visit_occurrence"]:
            concept_col, source_value_col, source_concept_col = MAPPING_CONFIG[table]
            if not table_exists(conn, table):
                continue

            cols = columns_for(conn, table)
            source_value_col = source_value_col if source_value_col in cols else None
            source_concept_col = source_concept_col if source_concept_col in cols else None

            if concept_col not in cols:
                continue

            total = safe_count(conn, table)
            unmapped = int(conn.execute(
                f'SELECT COUNT(*) FROM "{table}" WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0'
            ).fetchone()[0])

            row = {
                "omop_table": table,
                "total_records": total,
                "unmapped_records": unmapped,
                "has_source_value_column": source_value_col is not None,
                "has_source_concept_column": source_concept_col is not None,
                "concept_vocabulary_loaded": has_concept,
            }

            if source_value_col:
                n, blank, uniq = conn.execute(
                    f'''SELECT COUNT(*),
                               SUM(CASE WHEN "{source_value_col}" IS NULL OR TRIM(CAST("{source_value_col}" AS TEXT)) = '' THEN 1 ELSE 0 END),
                               COUNT(DISTINCT CASE WHEN "{source_value_col}" IS NOT NULL AND TRIM(CAST("{source_value_col}" AS TEXT)) <> '' THEN "{source_value_col}" END)
                        FROM "{table}"
                        WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0'''
                ).fetchone()
                row["unmapped_blank_source_value_records"] = int(blank or 0)
                row["unmapped_unique_nonblank_source_values"] = int(uniq or 0)

                top = pd.read_sql_query(
                    f'''SELECT CAST("{source_value_col}" AS TEXT) AS source_value, COUNT(*) AS records
                        FROM "{table}"
                        WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0
                        GROUP BY CAST("{source_value_col}" AS TEXT)
                        ORDER BY records DESC
                        LIMIT 100''',
                    conn
                )
                if not top.empty:
                    save_private(top, f"{table}_top_unmapped_source_values_PRIVATE.csv")

            if source_concept_col:
                zero_sc, nonzero_sc = conn.execute(
                    f'''SELECT
                           SUM(CASE WHEN "{source_concept_col}" IS NULL OR "{source_concept_col}" = 0 THEN 1 ELSE 0 END),
                           SUM(CASE WHEN "{source_concept_col}" IS NOT NULL AND "{source_concept_col}" <> 0 THEN 1 ELSE 0 END)
                        FROM "{table}"
                        WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0'''
                ).fetchone()

                row["unmapped_zero_or_null_source_concept_id_records"] = int(zero_sc or 0)
                row["unmapped_nonzero_source_concept_id_records"] = int(nonzero_sc or 0)

                if has_concept:
                    concept_resolution = pd.read_sql_query(
                        f'''SELECT
                                t."{source_concept_col}" AS source_concept_id,
                                COUNT(*) AS records,
                                c.concept_name,
                                c.domain_id,
                                c.vocabulary_id,
                                c.standard_concept,
                                c.invalid_reason
                            FROM "{table}" t
                            LEFT JOIN concept c
                              ON c.concept_id = t."{source_concept_col}"
                            WHERE (t."{concept_col}" IS NULL OR t."{concept_col}" = 0)
                              AND t."{source_concept_col}" IS NOT NULL
                              AND t."{source_concept_col}" <> 0
                            GROUP BY t."{source_concept_col}", c.concept_name, c.domain_id,
                                     c.vocabulary_id, c.standard_concept, c.invalid_reason
                            ORDER BY records DESC
                            LIMIT 100''',
                        conn
                    )
                    if not concept_resolution.empty:
                        save_private(
                            concept_resolution,
                            f"{table}_unmapped_source_concept_resolution_PRIVATE.csv"
                        )

            signals = []
            if row.get("unmapped_blank_source_value_records", 0) > 0:
                signals.append("some unmapped rows have blank source values")
            if row.get("unmapped_zero_or_null_source_concept_id_records", 0) > 0:
                signals.append("some unmapped rows have zero/null source_concept_id")
            if row.get("unmapped_nonzero_source_concept_id_records", 0) > 0:
                signals.append("some unmapped rows have non-zero source_concept_id but zero target concept")
            row["diagnostic_signals_not_root_cause"] = "; ".join(signals) if signals else "no simple signal identified"
            gap_public_rows.append(row)

        conn.close()

gap_public_df = pd.DataFrame(gap_public_rows)
display(gap_public_df)
if not gap_public_df.empty:
    save_public(gap_public_df, "drug_visit_mapping_gap_diagnostic_summary.csv")

""


## 9. Inspect relevant FHIR medication and encounter coding

This helps determine whether source coding information exists upstream. Code-level distributions are stored privately; coding-system aggregates are GitHub-safe.

In [10]:
coding_rows = []

def add_coding(resource_type, location, coding):
    coding_rows.append({
        "resource_type": resource_type,
        "location": location,
        "system": coding.get("system"),
        "code": coding.get("code"),
        "display": coding.get("display"),
    })

with Timer("fhir_coding_inventory"):
    if not fhir_files:
        print("Skipped: no FHIR files.")
    else:
        for path in fhir_files:
            for res in iter_resources_from_file(path):
                rt = res.get("resourceType")

                if rt in {"MedicationRequest", "MedicationStatement"}:
                    cc = res.get("medicationCodeableConcept")
                    for c in flatten_codings(cc):
                        add_coding(rt, "medicationCodeableConcept", c)

                elif rt == "Medication":
                    for c in flatten_codings(res.get("code")):
                        add_coding(rt, "code", c)

                elif rt == "Encounter":
                    enc_class = res.get("class")
                    if isinstance(enc_class, dict):
                        if "coding" in enc_class:
                            for c in flatten_codings(enc_class):
                                add_coding(rt, "class", c)
                        elif "code" in enc_class:
                            add_coding(rt, "class", enc_class)
                    elif isinstance(enc_class, list):
                        for item in enc_class:
                            if isinstance(item, dict) and "coding" in item:
                                for c in flatten_codings(item):
                                    add_coding(rt, "class", c)
                            elif isinstance(item, dict) and "code" in item:
                                add_coding(rt, "class", item)

                    for idx, typ in enumerate(res.get("type", []) or []):
                        for c in flatten_codings(typ):
                            add_coding(rt, f"type[{idx}]", c)

fhir_coding_df = pd.DataFrame(coding_rows)

if not fhir_coding_df.empty:
    system_summary = (
        fhir_coding_df
        .groupby(["resource_type", "location", "system"], dropna=False)
        .agg(coding_instances=("code", "size"), unique_codes=("code", "nunique"))
        .reset_index()
        .sort_values(["resource_type", "coding_instances"], ascending=[True, False])
    )
    display(system_summary)
    save_public(system_summary, "fhir_medication_encounter_coding_system_summary.csv")

    code_detail = (
        fhir_coding_df
        .groupby(["resource_type", "location", "system", "code", "display"], dropna=False)
        .size().reset_index(name="occurrences")
        .sort_values("occurrences", ascending=False)
    )
    save_private(code_detail, "fhir_medication_encounter_code_distribution_PRIVATE.csv")
else:
    print("No relevant coding extracted.")

Skipped: no FHIR files.
[timing] fhir_coding_inventory: 0.000 sec
No relevant coding extracted.


## 10. Inspect pyOMOP source paths for Encounter/drug mapping

This avoids guessing. The public file only lists match locations; surrounding source snippets are kept in the private diagnostics directory.

In [11]:
import pyomop

pyomop_root = Path(inspect.getfile(pyomop)).resolve().parent
print("pyOMOP source root:", pyomop_root)

search_terms = [
    "visit_concept_id",
    "visit_source_value",
    "visit_source_concept_id",
    "Encounter",
    "drug_concept_id",
    "drug_source_value",
    "MedicationRequest",
]

matches = []
snippet_blocks = []

for pyfile in pyomop_root.rglob("*.py"):
    try:
        lines = pyfile.read_text(encoding="utf-8", errors="ignore").splitlines()
    except Exception:
        continue

    for i, line in enumerate(lines, start=1):
        for term in search_terms:
            if term.lower() in line.lower():
                matches.append({
                    "term": term,
                    "file": str(pyfile.relative_to(pyomop_root)),
                    "line_number": i,
                })
                start = max(1, i - 3)
                end = min(len(lines), i + 3)
                block = "\n".join(f"{j:04d}: {lines[j-1]}" for j in range(start, end + 1))
                snippet_blocks.append(
                    f"term={term} | file={pyfile.relative_to(pyomop_root)} | line={i}\n{block}\n"
                )

pyomop_match_df = pd.DataFrame(matches).drop_duplicates()
display(pyomop_match_df.head(50))

if not pyomop_match_df.empty:
    save_public(pyomop_match_df, "pyomop_mapping_source_match_index.csv")
    snippet_path = PRIVATE_RESULTS_DIR / "pyomop_mapping_source_snippets_PRIVATE.txt"
    snippet_path.write_text("\n\n".join(snippet_blocks), encoding="utf-8")
    print("Saved PRIVATE source snippets:", snippet_path)

pyOMOP source root: /usr/local/lib/python3.12/dist-packages/pyomop


,term,file,line_number
0,Encounter,loader.py,62
1,drug_concept_id,cdm54/cdm54_tables.py,330
2,drug_concept_id,cdm54/cdm54_tables.py,360
3,visit_concept_id,cdm54/cdm54_tables.py,795
4,visit_source_value,cdm54/cdm54_tables.py,803
5,visit_source_concept_id,cdm54/cdm54_tables.py,804
6,visit_concept_id,cdm54/cdm54_tables.py,838
7,visit_source_concept_id,cdm54/cdm54_tables.py,841
8,drug_concept_id,cdm54/cdm54_tables.py,1004
9,drug_source_value,cdm54/cdm54_tables.py,1029


Saved public: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/pyomop_mapping_source_match_index.csv
Saved PRIVATE source snippets: /content/OHDSI_PRIVATE_REVIEWER_DIAGNOSTICS/pyomop_mapping_source_snippets_PRIVATE.txt


# Part C — Automated checks and practical performance

## 11. Automated validation

In [12]:
if resolved_db and Path(resolved_db).exists():
    conn = sqlite3.connect(str(resolved_db))

    for table in [
        "person", "visit_occurrence", "condition_occurrence",
        "drug_exposure", "measurement", "observation"
    ]:
        exists = table_exists(conn, table)
        add_validation(
            f"TBL-{table}",
            f"Required OMOP table exists: {table}",
            exists,
            True,
            "PASS" if exists else "FAIL"
        )
        if exists:
            count = safe_count(conn, table)
            ref = REFERENCE.get(table)
            add_validation(
                f"COUNT-{table}",
                f"{table} row count vs accepted reference",
                count,
                ref,
                "PASS" if count == ref else "REVIEW",
                "Reference comparison only; a justified correction may change the count."
            )

    if table_exists(conn, "concept"):
        concept_count = safe_count(conn, "concept")
        add_validation(
            "VOC-001",
            "OMOP concept table loaded",
            concept_count,
            ">0",
            "PASS" if concept_count and concept_count > 0 else "FAIL"
        )
    else:
        add_validation(
            "VOC-001",
            "OMOP concept table loaded",
            False,
            True,
            "FAIL",
            "Mapping interpretation is limited without vocabulary."
        )

    conn.close()

validation_df = pd.DataFrame(validation_rows)
display(validation_df)
save_public(validation_df, "automated_validation_results.csv")

""


Saved public: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/automated_validation_results.csv


PosixPath('/content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/automated_validation_results.csv')

## 12. Time the existing showcase analytics

The repository's `run_showcase_demo.sh` is timed separately and labeled **analytics/demo-script runtime**, not full transformation runtime.

The cell symlinks the private SQLite database into the path expected by the repository scripts, avoiding a large file copy.

In [13]:
analytics_timing_row = None

if resolved_db and Path(resolved_db).exists():
    expected_db = REPO_DIR / "data" / "omop_db" / "ohdsi_demo.sqlite"
    expected_db.parent.mkdir(parents=True, exist_ok=True)

    try:
        if expected_db.exists() or expected_db.is_symlink():
            try:
                same = expected_db.resolve() == Path(resolved_db).resolve()
            except Exception:
                same = False
            if not same:
                expected_db.unlink()
        if not expected_db.exists():
            expected_db.symlink_to(Path(resolved_db).resolve())
        print("DB link:", expected_db, "=>", expected_db.resolve())
    except Exception as e:
        print("Could not create DB symlink:", e)

    demo_script = REPO_DIR / "run_showcase_demo.sh"
    if demo_script.exists() and expected_db.exists():
        start = time.perf_counter()
        proc = subprocess.run(
            ["bash", str(demo_script)],
            cwd=str(REPO_DIR),
            capture_output=True,
            text=True
        )
        elapsed = time.perf_counter() - start

        analytics_timing_row = {
            "stage": "repository_showcase_demo_scripts",
            "elapsed_seconds": round(elapsed, 4),
            "return_code": proc.returncode,
            "scope": "analytics/demo scripts only; NOT full FHIR-to-OMOP transformation",
        }
        display(pd.DataFrame([analytics_timing_row]))
        save_public(pd.DataFrame([analytics_timing_row]), "analytics_demo_script_timing.csv")

        log_path = PRIVATE_RESULTS_DIR / "showcase_demo_timing_stdout_stderr_PRIVATE.txt"
        log_path.write_text(
            "STDOUT\n" + proc.stdout + "\n\nSTDERR\n" + proc.stderr,
            encoding="utf-8"
        )
        print("stdout tail:\n", proc.stdout[-3000:])
        if proc.stderr.strip():
            print("stderr tail:\n", proc.stderr[-3000:])
    else:
        print("Skipped: demo script or DB link unavailable.")
else:
    print("Skipped: OMOP database unavailable.")

Skipped: OMOP database unavailable.


## 13. Optional: time the exact full FHIR→OMOP transformation command

Set `TRANSFORM_COMMAND` only after the exact transformation command is known. This captures elapsed time and maximum resident memory where available.

Do **not** report analytics-script timing as full ETL timing.

In [14]:
full_transform_result = None

if TRANSFORM_COMMAND.strip():
    time_bin = Path("/usr/bin/time")
    wrapped = [str(time_bin), "-v", "bash", "-lc", TRANSFORM_COMMAND] if time_bin.exists() else ["bash", "-lc", TRANSFORM_COMMAND]

    start = time.perf_counter()
    proc = subprocess.run(
        wrapped,
        cwd=str(REPO_DIR),
        capture_output=True,
        text=True
    )
    elapsed = time.perf_counter() - start

    max_rss_kb = None
    m = re.search(r"Maximum resident set size \(kbytes\):\s*(\d+)", proc.stderr)
    if m:
        max_rss_kb = int(m.group(1))

    full_transform_result = {
        "stage": "full_fhir_to_omop_transform_command",
        "elapsed_seconds": round(elapsed, 4),
        "max_rss_mb": round(max_rss_kb / 1024, 2) if max_rss_kb is not None else np.nan,
        "return_code": proc.returncode,
        "scope": "single runtime environment; practical performance only",
        "command_recorded": TRANSFORM_COMMAND,
    }
    full_transform_timing_df = pd.DataFrame([full_transform_result])
    display(full_transform_timing_df)
    save_public(full_transform_timing_df, "full_transform_practical_timing.csv")

    log_path = PRIVATE_RESULTS_DIR / "full_transform_timing_stdout_stderr_PRIVATE.txt"
    log_path.write_text(
        "COMMAND\n" + TRANSFORM_COMMAND + "\n\nSTDOUT\n" + proc.stdout + "\n\nSTDERR\n" + proc.stderr,
        encoding="utf-8"
    )
else:
    print("TRANSFORM_COMMAND is blank. Full transformation timing intentionally skipped.")

TRANSFORM_COMMAND is blank. Full transformation timing intentionally skipped.


# Part D — Reviewer-facing synthesis

## 14. Build evidence matrix and Markdown summary

In [15]:
evidence_rows = []

if not person_reconciliation_summary.empty:
    evidence_rows.append({
        "reviewer_issue": "25 FHIR patients vs 27 OMOP PERSON rows",
        "evidence_generated": "person_reconciliation_summary.csv + private ID diagnostics",
        "current_result": person_reconciliation_summary.to_json(orient="records"),
        "interpretation_status": "Final cause requires verified record-level lineage"
    })

if not mapping_df.empty:
    drug = mapping_df[mapping_df["omop_table"] == "drug_exposure"]
    if not drug.empty and pd.notna(drug.iloc[0].get("total_records")):
        d = drug.iloc[0]
        evidence_rows.append({
            "reviewer_issue": "drug_exposure mapping gap",
            "evidence_generated": "mapping + drug/visit diagnostic summaries",
            "current_result": f"{int(d['mapped_records']):,}/{int(d['total_records']):,} mapped = {float(d['mapped_percent']):.2f}%",
            "interpretation_status": "Root cause must use source/FHIR/pyOMOP evidence"
        })

    visit = mapping_df[mapping_df["omop_table"] == "visit_occurrence"]
    if not visit.empty and pd.notna(visit.iloc[0].get("total_records")):
        v = visit.iloc[0]
        evidence_rows.append({
            "reviewer_issue": "visit_occurrence concept mapping",
            "evidence_generated": "visit statement + private visit diagnostics",
            "current_result": f"{int(v['total_records']):,} visit rows; {int(v['mapped_records']):,} non-zero visit_concept_id = {float(v['mapped_percent']):.2f}%",
            "interpretation_status": "Row creation and visit terminology mapping are distinct"
        })

if analytics_timing_row:
    evidence_rows.append({
        "reviewer_issue": "execution time",
        "evidence_generated": "analytics_demo_script_timing.csv + environment_summary.csv",
        "current_result": f"{analytics_timing_row['elapsed_seconds']:.4f} sec for showcase analytics scripts",
        "interpretation_status": "Not full transformation timing"
    })

if full_transform_result:
    evidence_rows.append({
        "reviewer_issue": "full transformation practical timing",
        "evidence_generated": "full_transform_practical_timing.csv + environment_summary.csv",
        "current_result": f"{full_transform_result['elapsed_seconds']:.4f} sec",
        "interpretation_status": "Single-environment practical performance; not a benchmark"
    })

reviewer_evidence_df = pd.DataFrame(evidence_rows)
display(reviewer_evidence_df)
save_public(reviewer_evidence_df, "reviewer_evidence_matrix.csv")

def fmt_num(x):
    try:
        return f"{int(x):,}"
    except Exception:
        return "NA"

lines = [
    "# OHDSI FHIR-to-OMOP Reviewer Hardening Summary",
    "",
    "This summary was generated from the reviewer-diagnostic Google Colab notebook.",
    "",
    "## Scope",
    "",
    "The analysis strengthens the accepted lightweight educational/software-demonstration workflow. "
    "It is not presented as a production ETL benchmark or comparative validation study.",
    "",
    "## PERSON reconciliation",
    "",
]

if not person_reconciliation_summary.empty:
    pr = person_reconciliation_summary.iloc[0].to_dict()
    lines += [
        f"- FHIR Patient resources observed: **{fmt_num(pr.get('fhir_patient_resources'))}**",
        f"- Unique FHIR Patient IDs observed: **{fmt_num(pr.get('fhir_unique_patient_ids'))}**",
        f"- OMOP PERSON rows observed: **{fmt_num(pr.get('omop_person_rows'))}**",
    ]
    if "matched_unique_ids" in pr:
        lines += [
            f"- Matched unique IDs: **{fmt_num(pr.get('matched_unique_ids'))}**",
            f"- OMOP-only unique IDs: **{fmt_num(pr.get('omop_only_unique_ids'))}**",
            f"- FHIR-only unique IDs: **{fmt_num(pr.get('fhir_only_unique_ids'))}**",
            "",
            "The final explanation should be stated only after the private identifier-level diagnostics "
            "and transformation behavior are inspected."
        ]
else:
    lines.append("PERSON reconciliation was not executed because required assets were unavailable.")

lines += ["", "## Mapping completeness", ""]
if not mapping_df.empty:
    for _, r in mapping_df.iterrows():
        if pd.notna(r.get("mapped_percent")):
            lines.append(
                f"- `{r['omop_table']}.{r['concept_column']}`: "
                f"**{float(r['mapped_percent']):.2f}%** "
                f"({fmt_num(r['mapped_records'])}/{fmt_num(r['total_records'])})"
            )
else:
    lines.append("Mapping completeness was not computed.")

lines += ["", "## Visit mapping interpretation", ""]
if not visit_statement_df.empty:
    lines.append("- " + visit_statement_df.iloc[0]["recommended_interpretation"])
    lines.append("- Creation of visit rows and mapping of `visit_concept_id` are separate outcomes.")
else:
    lines.append("Visit statement was not generated.")

lines += ["", "## Practical performance", ""]
if full_transform_result:
    lines.append(
        f"- Full configured transformation command: **{full_transform_result['elapsed_seconds']:.3f} seconds**."
    )
    lines.append("- Single-environment practical performance; not a production benchmark.")
elif analytics_timing_row:
    lines.append(
        f"- Existing showcase analytics scripts: **{analytics_timing_row['elapsed_seconds']:.3f} seconds**."
    )
    lines.append("- This is analytics/demo-script timing only, not full transformation time.")
else:
    lines.append("Timing was not available in this run.")

lines += [
    "",
    "## Root-cause rule",
    "",
    "Drug and visit root causes should be written only after OMOP source values/source concept identifiers, "
    "raw FHIR coding, and the relevant pyOMOP implementation have been inspected together.",
]

summary_path = PUBLIC_RESULTS_DIR / "OHDSI_REVIEWER_HARDENING_SUMMARY.md"
summary_path.write_text("\n".join(lines), encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))

""


Saved public: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/reviewer_evidence_matrix.csv
# OHDSI FHIR-to-OMOP Reviewer Hardening Summary

This summary was generated from the reviewer-diagnostic Google Colab notebook.

## Scope

The analysis strengthens the accepted lightweight educational/software-demonstration workflow. It is not presented as a production ETL benchmark or comparative validation study.

## PERSON reconciliation

PERSON reconciliation was not executed because required assets were unavailable.

## Mapping completeness

Mapping completeness was not computed.

## Visit mapping interpretation

Visit statement was not generated.

## Practical performance

Timing was not available in this run.

## Root-cause rule

Drug and visit root causes should be written only after OMOP source values/source concept identifiers, raw FHIR coding, and the relevant pyOMOP implementation have been inspected together.


## 15. Package public outputs and inspect Git status

Only aggregate outputs are packaged. Private diagnostics remain outside the repository.

In [16]:
if timings:
    stage_timing_df = pd.DataFrame(
        [{"stage": k, "elapsed_seconds": round(v, 6)} for k, v in timings.items()]
    )
    display(stage_timing_df)
    save_public(stage_timing_df, "notebook_stage_timing_registry.csv")

public_readme = PUBLIC_RESULTS_DIR / "README.md"
public_readme.write_text(
    "# Reviewer Hardening Outputs\n\n"
    "Generated by `OHDSI_Reviewer_Driven_Hardening_COLAB.ipynb`.\n\n"
    "These files contain aggregate/reviewer-facing summaries intended for version control.\n\n"
    "Do **not** copy `/content/OHDSI_PRIVATE_REVIEWER_DIAGNOSTICS/` into GitHub without manual review "
    "of row-level identifiers, source values, logs, and source-code snippets.\n\n"
    "Athena vocabulary distribution files are not included.\n",
    encoding="utf-8"
)

archive_path = shutil.make_archive(
    "/content/OHDSI_reviewer_hardening_public_outputs",
    "zip",
    root_dir=str(PUBLIC_RESULTS_DIR)
)
print("Public output ZIP:", archive_path)

print("\nRepository git status:")
subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], check=False)

print("\nPublic files:")
for p in sorted(PUBLIC_RESULTS_DIR.glob("*")):
    if p.is_file():
        print(" -", p.relative_to(REPO_DIR))

print("\nPrivate diagnostics — DO NOT AUTO-PUSH:")
for p in sorted(PRIVATE_RESULTS_DIR.glob("*")):
    if p.is_file():
        print(" -", p)

,stage,elapsed_seconds
0,fhir_inventory,0.000046
1,fhir_coding_inventory,0.000067


Saved public: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_hardening/notebook_stage_timing_registry.csv
Public output ZIP: /content/OHDSI_reviewer_hardening_public_outputs.zip

Repository git status:

Public files:
 - results/reviewer_hardening/OHDSI_REVIEWER_HARDENING_SUMMARY.md
 - results/reviewer_hardening/README.md
 - results/reviewer_hardening/automated_validation_results.csv
 - results/reviewer_hardening/environment_summary.csv
 - results/reviewer_hardening/fhir_resource_counts.csv
 - results/reviewer_hardening/notebook_stage_timing_registry.csv
 - results/reviewer_hardening/pyomop_mapping_source_match_index.csv
 - results/reviewer_hardening/reviewer_evidence_matrix.csv

Private diagnostics — DO NOT AUTO-PUSH:
 - /content/OHDSI_PRIVATE_REVIEWER_DIAGNOSTICS/pyomop_mapping_source_snippets_PRIVATE.txt
